In [ ]:
import os
import csv
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.metrics import r2_score
from torch.cuda.amp import GradScaler, autocast

# Random seed configuration
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Model definitions
class SimpleGNN(nn.Module):
    """Pretrained GCN teacher (original architecture)."""
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        if edge_dim:
            self.edge_norm = nn.BatchNorm1d(edge_dim)
        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]), nn.ReLU(), nn.Dropout(dropout)
            )
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h
        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        if hasattr(self, 'edge_norm') and data.edge_attr is not None:
            _ = self.edge_norm(data.edge_attr)
        u = getattr(data, 'u', None)
        if hasattr(self, 'global_norm') and u is not None:
            u = self.global_norm(u)
        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)
        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if u is not None else node_pool
        out = self.output_mlp(h).squeeze()
        return (out, h) if return_feat else out


class EnhancedGNN(nn.Module):
    """Student GCN used for distillation and inference."""
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None
        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]), nn.ReLU(), nn.Dropout(dropout)
            )
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h
        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim//2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        if self.edge_norm and hasattr(data, 'edge_attr') and data.edge_attr is not None:
            _ = self.edge_norm(data.edge_attr)
        u = getattr(data, 'u', None)
        gf = None
        if u is not None and self.global_norm is not None:
            u = self.global_norm(u)
            gf = self.global_mlp(u)
        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)
        pooled = global_mean_pool(x, data.batch)
        h = torch.cat([pooled, gf], dim=1) if gf is not None else pooled
        out = self.output_mlp(h).squeeze()
        return (out, h) if return_feat else out

class Adapter(nn.Module):
    """Project student graph features into the teacher feature space."""
    def __init__(self, dim_s, dim_t):
        super().__init__()
        self.linear = nn.Linear(dim_s, dim_t)
    def forward(self, h):
        return self.linear(h)

# Molecular graph mini-batches
def create_data_loader(graph_list, batch_size=32, shuffle=True):
    data_list = []
    for g in graph_list:
        data_list.append(Data(
            x=g['x'], edge_index=g['edge_index'],
            edge_attr=g.get('edge_attr', None), u=g.get('u', None),
            y=g['y'], y_soft=g.get('y_soft', None)
        ))
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

# Distillation training with fixed, uniform teacher weights
def train_model(
    train_dir, val_dir, teacher_paths, save_path,
    hint_lambda=5.0, weight_ratio=(0.6, 0.4),
    hidden_dims=(128, 128), dropout=0.1,
    epochs=500, batch_size=64, lr=1e-3, min_lr=1e-4,
    lr_patience=20, es_patience=50, seed=42,
):
    """Train one student using the arithmetic mean of all five teacher outputs."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_seed(seed)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Standardize hard labels using training-set statistics only.
    train_graphs = torch.load(os.path.join(train_dir, 'graph_data.pt'))
    val_graphs = torch.load(os.path.join(val_dir, 'graph_data.pt'))
    if not train_graphs or not val_graphs:
        raise ValueError('Both training and validation graph datasets must be nonempty.')
    K = len(teacher_paths)
    if K != 5:
        raise ValueError('This experiment requires exactly five pretrained teachers.')
    for split_name, graphs in (('training', train_graphs), ('validation', val_graphs)):
        for index, graph in enumerate(graphs):
            if 'y_soft' not in graph:
                raise ValueError(f'Missing teacher soft labels: {split_name} graph {index}.')
            soft = graph['y_soft']
            if soft.numel() != K or soft.dim() not in (1, 2):
                raise ValueError(f'Expected five teacher soft labels: {split_name} graph {index}.')

    hard_labels = torch.stack([graph['y'] for graph in train_graphs]).view(-1)
    y_mean = hard_labels.mean().item()
    y_std = hard_labels.std().item() + 1e-8
    for graph in train_graphs:
        graph['y'] = (graph['y'] - y_mean) / y_std
    for graph in val_graphs:
        graph['y'] = (graph['y'] - y_mean) / y_std

    train_loader = create_data_loader(train_graphs, batch_size, shuffle=True)
    val_loader = create_data_loader(val_graphs, batch_size, shuffle=False)

    sample = train_graphs[0]
    node_dim = sample['x'].size(1)
    edge_dim = sample['edge_attr'].size(1) if sample.get('edge_attr') is not None else 0
    global_dim = sample['u'].size(1) if sample.get('u') is not None else 0
    student = EnhancedGNN(node_dim, edge_dim, global_dim, hidden_dims, dropout).to(device)

    # Checkpoint order must match the five columns of the precomputed y_soft tensor.
    teachers = []
    for checkpoint_path in teacher_paths:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        teacher = SimpleGNN(
            checkpoint['node_dim'], checkpoint.get('edge_dim', 0),
            checkpoint.get('global_dim', 0), checkpoint['hidden_dims'],
            checkpoint['dropout']
        ).to(device)
        teacher.load_state_dict(checkpoint['model_state_dict'], strict=False)
        teacher.eval()
        for param in teacher.parameters():
            param.requires_grad_(False)
        teachers.append(teacher)

    adapter = Adapter(student.final_dim, student.final_dim).to(device)
    optimizer = optim.Adam(
        list(student.parameters()) + list(adapter.parameters()),
        lr=lr, weight_decay=1e-5
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=0.5, patience=lr_patience, min_lr=min_lr
    )
    scaler = GradScaler(enabled=(device.type == 'cuda'))
    best_val_r2, patience = -float('inf'), 0
    history = {'loss': [], 'val_r2': []}

    def validation_r2():
        """Measure validation R² for checkpoint selection (not final test reporting)."""
        student.eval()
        targets, predictions = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                prediction = student(batch).view(-1)
                targets.append(batch.y.view(-1).cpu().numpy())
                predictions.append(prediction.cpu().numpy())
        return r2_score(np.concatenate(targets), np.concatenate(predictions))

    for epoch in range(1, epochs + 1):
        student.train()
        running_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            with autocast(enabled=(device.type == 'cuda')):
                pred_s, h_s = student(batch, return_feat=True)
                pred_s = pred_s.view(-1)
                with torch.no_grad():
                    teacher_features = torch.stack(
                        [teacher(batch, return_feat=True)[1] for teacher in teachers], dim=1
                    )
                # Arithmetic mean: every teacher contributes exactly 1 / K.
                mean_teacher_features = teacher_features.mean(dim=1)
                loss_hint = F.mse_loss(adapter(h_s), mean_teacher_features)

                if batch.y_soft.dim() != 2 or batch.y_soft.size(1) != K:
                    raise ValueError('Expected batched soft labels with shape [batch_size, 5].')
                mean_soft_label = batch.y_soft.mean(dim=1)
                pred_f = weight_ratio[0] * pred_s + weight_ratio[1] * mean_soft_label
                # Preserve the original hard/soft/feature loss formulation.
                loss = (
                    weight_ratio[0] * F.mse_loss(pred_f, batch.y.view(-1))
                    + weight_ratio[1] * F.mse_loss(pred_s, mean_soft_label)
                    + hint_lambda * loss_hint
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)
        current_val_r2 = validation_r2()
        history['loss'].append(epoch_loss)
        history['val_r2'].append(current_val_r2)
        scheduler.step(epoch_loss)

        if current_val_r2 > best_val_r2:
            best_val_r2, patience = current_val_r2, 0
            torch.save({
                'model_state_dict': student.state_dict(),
                'adapter_state': adapter.state_dict(),
                'y_mean': y_mean, 'y_std': y_std,
                'history': history,
                'node_dim': node_dim, 'edge_dim': edge_dim,
                'global_dim': global_dim, 'hidden_dims': hidden_dims,
                'dropout': dropout, 'teacher_weighting': 'uniform',
                'num_teachers': K,
            }, save_path)
        else:
            patience += 1
            if patience >= es_patience:
                print(f'Early stopping at epoch {epoch}.')
                break

    print(f'Seed {seed}: checkpoint saved; best validation R² = {best_val_r2:.4f}')
    return save_path


# Main experiment: fixed-hyperparameter, uniformly weighted teacher distillation
if __name__ == '__main__':
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    PROJECT_ROOT = os.path.abspath(os.getcwd())
    config_path = os.path.join(PROJECT_ROOT, 'config', 'uniform_seed_hyperparameters.csv')

    # Values must come from the original uniform-weight experiment records.
    seed_configs = {}
    with open(config_path, newline='', encoding='utf-8') as config_file:
        for row in csv.DictReader(config_file):
            seed = int(row['seed'])
            if seed in seed_configs:
                raise ValueError(f'Duplicate hyperparameter entry for seed {seed}.')
            if not row['hint_lambda'] or not row['weight_student'] or not row['weight_teacher']:
                raise ValueError(f'Missing verified hyperparameters for seed {seed}.')
            hint_lambda = float(row['hint_lambda'])
            weight_ratio = (float(row['weight_student']), float(row['weight_teacher']))
            if hint_lambda < 0 or any(w < 0 for w in weight_ratio) or abs(sum(weight_ratio) - 1.0) > 1e-6:
                raise ValueError(f'Invalid distillation parameters for seed {seed}.')
            seed_configs[seed] = {
                'hint_lambda': hint_lambda,
                'weight_ratio': weight_ratio,
                'config_id': f'hl{hint_lambda:g}_wr{weight_ratio[0]:g}_{weight_ratio[1]:g}',
            }
    if set(seed_configs) != set(seeds):
        raise ValueError('The configuration file must contain exactly the specified ten seeds.')

    train_dir = os.path.join(PROJECT_ROOT, 'data-set', 'train')
    val_dir = os.path.join(PROJECT_ROOT, 'data-set', 'validation')
    teacher_order = ('qcut', 'elem', 'molwt', 'fp', 'scaffold')
    teacher_dir = os.path.join(PROJECT_ROOT, 'checkpoints', 'teachers')
    teacher_paths = [os.path.join(teacher_dir, f'{name}.pt') for name in teacher_order]

    save_root = os.path.join(PROJECT_ROOT, 'results', 'uniform_teacher_distillation')
    os.makedirs(save_root, exist_ok=True)
    epochs, batch_size = 1000, 64
    lr, min_lr = 1e-3, 5e-5
    lr_patience, es_patience = 30, 100
    hidden_dims, dropout = (128, 128), 0.1

    manifest_path = os.path.join(save_root, 'checkpoint_manifest.csv')
    with open(manifest_path, 'w', newline='', encoding='utf-8') as manifest_file:
        writer = csv.DictWriter(manifest_file, fieldnames=['seed', 'config_id', 'checkpoint_path'])
        writer.writeheader()
        for seed in seeds:
            config = seed_configs[seed]
            checkpoint_path = os.path.join(
                save_root, f"student_{config['config_id']}_seed{seed}.pt"
            )
            print(f"Training uniform-weight student with seed {seed}.")
            train_model(
                train_dir=train_dir,
                val_dir=val_dir,
                teacher_paths=teacher_paths,
                save_path=checkpoint_path,
                hint_lambda=config['hint_lambda'],
                weight_ratio=config['weight_ratio'],
                hidden_dims=hidden_dims,
                dropout=dropout,
                epochs=epochs,
                batch_size=batch_size,
                lr=lr,
                min_lr=min_lr,
                lr_patience=lr_patience,
                es_patience=es_patience,
                seed=seed,
            )
            writer.writerow({
                'seed': seed,
                'config_id': config['config_id'],
                'checkpoint_path': os.path.relpath(checkpoint_path, PROJECT_ROOT),
            })
            manifest_file.flush()
    print(f'Uniform-weight distillation complete. Manifest: {manifest_path}')
